# 14 — Scoring (BigQuery iterative PageRank)

**AWS-side equivalent:** Aurora SQL PageRank (`notebooks/aws/14-scoring.ipynb`)  
**GCP service:** BigQuery — iterative `WHILE` / `CREATE OR REPLACE TABLE` loop for PageRank,
plus Python Brandes for betweenness.

Same composite formula as AWS:

$$\text{composite} = 0.40 \cdot \widehat{\text{expertise}} + 0.20 \cdot \widehat{\text{centrality}} + 0.30 \cdot \widehat{\text{pagerank}} + 0.10 \cdot \widehat{\text{betweenness}}$$

where $\widehat{x}$ is min-max scaled to $[0, 1]$. The result is written back to
`people.composite_score`.

_As of 2026-05: BigQuery does **not** ship an `ML.PAGE_RANK` function — the iteration is
expressed directly with `WHILE` + `CREATE OR REPLACE TABLE`, mirroring the Aurora
recursive-CTE pattern. Betweenness still runs in Python (Brandes' algorithm) since BQ
has no built-in for it._

In [16]:
import json
import time
import datetime
from pathlib import Path
from collections import defaultdict, deque

from google.cloud import bigquery
from google.cloud.bigquery import QueryJobConfig

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
BQ_DATASET = gcp.get('bq_dataset') or gcp.get('bq_scoring_dataset', 'orgiq_scoring')

assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'

bq = bigquery.Client(project=PROJECT_ID)
DS = f'`{PROJECT_ID}.{BQ_DATASET}`'

n_people = next(iter(bq.query(
    f'SELECT COUNT(*) AS n FROM {DS}.people'
).result())).n
print(f'project={PROJECT_ID} dataset={BQ_DATASET} people={n_people}')

project=hoehncloudconsole dataset=orgiq people=20


## Step 1 — Build the unified edge table

Three relations contribute to expertise propagation. We union them, then de-dup on the
undirected key `(LEAST(a, b), GREATEST(a, b))` so each unordered pair appears exactly
once with summed weight:

| Source table | Edge meaning | Default weight |
|---|---|---|
| `collaborations` | Person ↔ Person (already undirected) | `weight` column |
| `project_contributions` | shared project → Person ↔ Person | 0.5 × min(weight_a, weight_b) |
| `decision_participants` | shared decision → Person ↔ Person | 0.3 × min(weight_a, weight_b) |

Output table: `{bq_dataset}._scoring_edges_uniq(person_a_id, person_b_id, weight)` — same
name as the AWS scratch table so cross-cloud diffs stay quiet.

In [ ]:
# Discover which edge-source tables are present — the BQ orgiq dataset may only
# have the core tables (people / skills / person_skills) without the graph tables.
existing_tables = {t.table_id for t in bq.list_tables(BQ_DATASET)}
print(f'Tables in {BQ_DATASET}: {sorted(existing_tables)}')

_parts = []
_used  = []

if 'collaborations' in existing_tables:
    _parts.append(f"""
  SELECT person_a_id AS a, person_b_id AS b, CAST(weight AS FLOAT64) AS w
    FROM {DS}.collaborations""")
    _used.append('collaborations')

if 'project_contributions' in existing_tables:
    _parts.append(f"""
  SELECT a.person_id AS a, b.person_id AS b,
         0.5 * LEAST(CAST(a.weight AS FLOAT64), CAST(b.weight AS FLOAT64)) AS w
    FROM {DS}.project_contributions a
    JOIN {DS}.project_contributions b USING (project_id)
   WHERE a.person_id <> b.person_id""")
    _used.append('project_contributions')

if 'decision_participants' in existing_tables:
    _parts.append(f"""
  SELECT a.person_id AS a, b.person_id AS b,
         0.3 * LEAST(CAST(a.weight AS FLOAT64), CAST(b.weight AS FLOAT64)) AS w
    FROM {DS}.decision_participants a
    JOIN {DS}.decision_participants b USING (decision_id)
   WHERE a.person_id <> b.person_id""")
    _used.append('decision_participants')

SKILL_FALLBACK = f"""
  SELECT ps_a.person_id AS a, ps_b.person_id AS b,
         SUM(LEAST(CAST(ps_a.weight AS FLOAT64), CAST(ps_b.weight AS FLOAT64))) AS w
    FROM {DS}.person_skills ps_a
    JOIN {DS}.person_skills ps_b USING (skill_id)
   WHERE ps_a.person_id <> ps_b.person_id
   GROUP BY ps_a.person_id, ps_b.person_id"""

if not _parts:
    _parts.append(SKILL_FALLBACK)
    _used.append('person_skills (shared-skill fallback — no graph tables)')

print(f'Edge sources (first pass): {_used}')

EDGES_SQL = f"""
CREATE OR REPLACE TABLE {DS}._scoring_edges_uniq AS
WITH unioned AS ({" UNION ALL ".join(_parts)}
),
canon AS (
  SELECT
    LEAST(a, b)    AS person_a_id,
    GREATEST(a, b) AS person_b_id,
    w
  FROM unioned
  WHERE a IS NOT NULL AND b IS NOT NULL AND a <> b
)
SELECT person_a_id, person_b_id, SUM(w) AS weight
  FROM canon
 GROUP BY person_a_id, person_b_id;
"""

bq.query(EDGES_SQL, job_config=QueryJobConfig(use_query_cache=False)).result()
n_edges = next(iter(bq.query(
    f'SELECT COUNT(*) AS n FROM {DS}._scoring_edges_uniq'
).result())).n

# If graph tables were empty, fall back to skill-affinity edges.
if n_edges == 0 and SKILL_FALLBACK not in _parts:
    print('ℹ  Graph tables are empty — falling back to person_skills edges.')
    EDGES_SQL_FALLBACK = f"""
CREATE OR REPLACE TABLE {DS}._scoring_edges_uniq AS
WITH unioned AS ({SKILL_FALLBACK}
),
canon AS (
  SELECT
    LEAST(a, b)    AS person_a_id,
    GREATEST(a, b) AS person_b_id,
    w
  FROM unioned
  WHERE a IS NOT NULL AND b IS NOT NULL AND a <> b
)
SELECT person_a_id, person_b_id, SUM(w) AS weight
  FROM canon
 GROUP BY person_a_id, person_b_id;
"""
    bq.query(EDGES_SQL_FALLBACK, job_config=QueryJobConfig(use_query_cache=False)).result()
    n_edges = next(iter(bq.query(
        f'SELECT COUNT(*) AS n FROM {DS}._scoring_edges_uniq'
    ).result())).n
    _used.append('person_skills (shared-skill fallback — graph tables were empty)')

print(f'✓ _scoring_edges_uniq materialised: {n_edges} unique undirected edges')


## Step 2 — PageRank via iterative `WHILE` / `MERGE`

BigQuery has no `ML.PAGE_RANK` (despite the rumour mill) — we run the fixpoint by hand.
20 iterations with damping `d = 0.85`. Each undirected edge contributes in both directions
(`person_a_id → person_b_id` and vice versa) via a `UNION ALL` inside the loop body.

$$PR_{t+1}(v) = \frac{1-d}{N} + d \sum_{u \to v} \frac{w_{u,v}}{W_u} \cdot PR_t(u)$$

Each iteration writes a fresh `_pr` table. 20 rounds is enough for convergence on
graphs of a few thousand nodes.

In [18]:
PR_SQL = f"""
DECLARE i INT64 DEFAULT 0;
DECLARE d FLOAT64 DEFAULT 0.85;
DECLARE n INT64 DEFAULT (SELECT COUNT(*) FROM {DS}.people);

-- Directed view of the undirected edge table: each pair contributes both ways.
CREATE OR REPLACE TABLE {DS}._pr_edges AS
  SELECT person_a_id AS src, person_b_id AS dst, weight FROM {DS}._scoring_edges_uniq
  UNION ALL
  SELECT person_b_id AS src, person_a_id AS dst, weight FROM {DS}._scoring_edges_uniq;

-- Pre-aggregate weighted out-degree once — it's invariant across iterations.
CREATE OR REPLACE TABLE {DS}._pr_outw AS
  SELECT src, SUM(weight) AS w_total
    FROM {DS}._pr_edges
   GROUP BY src;

-- Seed: PR(v) = 1/N for every person.
CREATE OR REPLACE TABLE {DS}._pr AS
  SELECT id AS node_id, 1.0 / n AS pr FROM {DS}.people;

WHILE i < 20 DO
  CREATE OR REPLACE TABLE {DS}._pr AS
    SELECT p.id AS node_id,
           (1 - d) / n +
           d * COALESCE(SUM(e.weight / o.w_total * prev.pr), 0) AS pr
      FROM {DS}.people p
      LEFT JOIN {DS}._pr_edges e ON e.dst = p.id
      LEFT JOIN {DS}._pr_outw  o ON o.src = e.src
      LEFT JOIN {DS}._pr       prev ON prev.node_id = e.src
     GROUP BY p.id;
  SET i = i + 1;
END WHILE;
"""

t0 = time.time()
bq.query(PR_SQL, job_config=QueryJobConfig(use_query_cache=False)).result()
elapsed = time.time() - t0
print(f'✓ PageRank converged after 20 iterations ({elapsed:.1f}s)')

top_pr = list(bq.query(f"""
  SELECT p.id, p.name, _pr.pr
    FROM {DS}._pr
    JOIN {DS}.people p ON p.id = _pr.node_id
   ORDER BY _pr.pr DESC
   LIMIT 10
""").result())
print()
print(f"{'Rank':<5} {'Name':<28} {'PageRank':<14}")
print('-' * 50)
for i, row in enumerate(top_pr, 1):
    print(f'{i:<5} {row.name:<28} {row.pr:.8f}')

BadRequest: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/hoehncloudconsole/queries/093dccd8-47b5-47ee-b491-854d29309dcc?maxResults=0&location=US&prettyPrint=false: Exceeded rate limits: too many table update operations for this table. For more information, see https://docs.cloud.google.com/bigquery/troubleshooting-errors at [23:3]

Location: US
Job ID: 093dccd8-47b5-47ee-b491-854d29309dcc


## Step 3 — Expertise score

Sum of `person_skills.weight` per person. Simplest term in the composite, most legible
to users. Stored in a scratch table so the final `MERGE` joins are uniform.

In [ ]:
EXP_SQL = f"""
CREATE OR REPLACE TABLE {DS}._expertise AS
  SELECT person_id, CAST(SUM(weight) AS FLOAT64) AS expertise
    FROM {DS}.person_skills
   GROUP BY person_id;
"""
bq.query(EXP_SQL, job_config=QueryJobConfig(use_query_cache=False)).result()

n_exp = next(iter(bq.query(
    f'SELECT COUNT(*) AS n FROM {DS}._expertise'
).result())).n
print(f'✓ _expertise computed for {n_exp} people')

## Step 4 — Centrality + Brandes betweenness in Python

BigQuery has no betweenness primitive and a SQL-only Brandes would be unreadable. We pull
the edge table into a pandas DataFrame, build an undirected adjacency in pure Python,
compute degree centrality and Brandes' BFS betweenness, then load both back into BQ as
`_python_scores(person_id, centrality, betweenness)`.

For graphs > 1k people we sample 1k source nodes and rescale (matches the AWS notebook).

In [ ]:
import random

# Use row iterators instead of .to_dataframe() — avoids the db_dtypes dependency.
edge_rows = list(bq.query(
    f'SELECT person_a_id, person_b_id, weight FROM {DS}._scoring_edges_uniq'
).result())
people_rows = list(bq.query(
    f'SELECT id FROM {DS}.people'
).result())
person_ids = [r['id'] for r in people_rows]
pid_set = set(person_ids)

adj = defaultdict(set)
for e in edge_rows:
    a, b = e['person_a_id'], e['person_b_id']
    if a in pid_set and b in pid_set:
        adj[a].add(b)
        adj[b].add(a)

centrality = {pid: float(len(adj[pid])) for pid in person_ids}

n = len(person_ids)
if n > 1000:
    sources = random.sample(person_ids, 1000)
    scale = n / 1000.0
    print(f'⚠️  Sampling 1000 of {n} sources for Brandes (scale={scale:.2f})')
else:
    sources = list(person_ids)
    scale = 1.0

betweenness = defaultdict(float)
for s in sources:
    stack = []
    pred = defaultdict(list)
    sigma = defaultdict(float)
    dist = defaultdict(lambda: -1)
    sigma[s] = 1.0
    dist[s] = 0
    queue = deque([s])
    while queue:
        v = queue.popleft()
        stack.append(v)
        for w in adj[v]:
            if dist[w] < 0:
                queue.append(w)
                dist[w] = dist[v] + 1
            if dist[w] == dist[v] + 1:
                sigma[w] += sigma[v]
                pred[w].append(v)
    delta = defaultdict(float)
    while stack:
        w = stack.pop()
        for v in pred[w]:
            delta[v] += (sigma[v] / sigma[w]) * (1.0 + delta[w])
        if w != s:
            betweenness[w] += delta[w]

norm = 1.0 / max(1, (n - 1) * (n - 2) / 2.0)
betweenness_scores = {pid: betweenness[pid] * norm * scale for pid in person_ids}

print(f'✓ centrality + betweenness computed for {n} people')

rows = [
    {
        'person_id': pid,
        'centrality': float(centrality[pid]),
        'betweenness': float(betweenness_scores[pid]),
    }
    for pid in person_ids
]

load_job = bq.load_table_from_json(
    rows,
    f'{PROJECT_ID}.{BQ_DATASET}._python_scores',
    job_config=bigquery.LoadJobConfig(
        write_disposition='WRITE_TRUNCATE',
        schema=[
            bigquery.SchemaField('person_id', 'STRING'),
            bigquery.SchemaField('centrality', 'FLOAT64'),
            bigquery.SchemaField('betweenness', 'FLOAT64'),
        ],
    ),
)
load_job.result()
print(f'✓ _python_scores loaded ({len(rows)} rows)')


## Step 5 — Compose and write back to `people`

Single `MERGE` joining `people` against `_pr`, `_expertise`, and `_python_scores`.
Min-max normalisation is computed inside a CTE (`bounds`) so the same statement that
reads the components also writes the normalised composite. Same weights as the AWS
notebook (0.40 / 0.20 / 0.30 / 0.10) so cross-cloud comparisons line up.

In [ ]:
# Add scoring columns to people if they don't already exist.
ALTER_SQL = f"""
ALTER TABLE {DS}.people
  ADD COLUMN IF NOT EXISTS expertise_score   FLOAT64,
  ADD COLUMN IF NOT EXISTS centrality_score  FLOAT64,
  ADD COLUMN IF NOT EXISTS pagerank_score    FLOAT64,
  ADD COLUMN IF NOT EXISTS betweenness_score FLOAT64,
  ADD COLUMN IF NOT EXISTS composite_score   FLOAT64,
  ADD COLUMN IF NOT EXISTS scored_at         TIMESTAMP;
"""
bq.query(ALTER_SQL).result()
print('✓ scoring columns present on people table')

# The PageRank WHILE loop issues 20+ DML ops; BQ enforces a rolling rate limit
# (~5 ops/10s per table). Pause so the window resets before the MERGE.
print('Waiting 15s for BQ DML rate-limit window to reset...')
time.sleep(15)

COMPOSE_SQL = f"""
MERGE {DS}.people AS p
USING (
  WITH joined AS (
    SELECT
      p.id AS person_id,
      COALESCE(_pr.pr, 0.0)              AS pagerank_raw,
      COALESCE(_exp.expertise, 0.0)      AS expertise_raw,
      COALESCE(_py.centrality, 0.0)      AS centrality_raw,
      COALESCE(_py.betweenness, 0.0)     AS betweenness_raw
    FROM {DS}.people p
    LEFT JOIN {DS}._pr            ON _pr.node_id   = p.id
    LEFT JOIN {DS}._expertise _exp ON _exp.person_id = p.id
    LEFT JOIN {DS}._python_scores _py ON _py.person_id = p.id
  ),
  bounds AS (
    SELECT
      MIN(expertise_raw)   AS e_lo, MAX(expertise_raw)   AS e_hi,
      MIN(centrality_raw)  AS c_lo, MAX(centrality_raw)  AS c_hi,
      MIN(pagerank_raw)    AS p_lo, MAX(pagerank_raw)    AS p_hi,
      MIN(betweenness_raw) AS b_lo, MAX(betweenness_raw) AS b_hi
    FROM joined
  )
  SELECT
    j.person_id,
    j.expertise_raw,
    j.centrality_raw,
    j.pagerank_raw,
    j.betweenness_raw,
    0.40 * COALESCE(SAFE_DIVIDE(j.expertise_raw   - b.e_lo, NULLIF(b.e_hi - b.e_lo, 0)), 0) +
    0.20 * COALESCE(SAFE_DIVIDE(j.centrality_raw  - b.c_lo, NULLIF(b.c_hi - b.c_lo, 0)), 0) +
    0.30 * COALESCE(SAFE_DIVIDE(j.pagerank_raw    - b.p_lo, NULLIF(b.p_hi - b.p_lo, 0)), 0) +
    0.10 * COALESCE(SAFE_DIVIDE(j.betweenness_raw - b.b_lo, NULLIF(b.b_hi - b.b_lo, 0)), 0)
      AS composite
  FROM joined j
  CROSS JOIN bounds b
) AS s
ON p.id = s.person_id
WHEN MATCHED THEN UPDATE SET
  expertise_score   = s.expertise_raw,
  centrality_score  = s.centrality_raw,
  pagerank_score    = s.pagerank_raw,
  betweenness_score = s.betweenness_raw,
  composite_score   = s.composite,
  scored_at         = CURRENT_TIMESTAMP();
"""

merge_job = bq.query(COMPOSE_SQL, job_config=QueryJobConfig(use_query_cache=False))
merge_job.result()
print(f'✓ MERGE applied to people ({merge_job.num_dml_affected_rows} rows updated)')


## Step 6 — Verify and snapshot

Pull the top 10 by `composite_score`, print as a DataFrame, then persist the snapshot
timestamp + row count to `config.json` so downstream notebooks (advisor, portal seeding)
can reference the latest run.

In [ ]:
leaderboard_rows = list(bq.query(f"""
  SELECT id, name,
         expertise_score,
         pagerank_score,
         composite_score
    FROM {DS}.people
   ORDER BY composite_score DESC NULLS LAST
   LIMIT 10
""").result())

print(f"{'id':<10} {'name':<28} {'expertise':>10} {'pagerank':>12} {'composite':>10}")
print('-' * 74)
for r in leaderboard_rows:
    print(
        f"{str(r['id']):<10} {str(r['name']):<28}"
        f" {(r['expertise_score'] or 0):>10.4f}"
        f" {(r['pagerank_score'] or 0):>12.8f}"
        f" {(r['composite_score'] or 0):>10.4f}"
    )

snapshot_ts = int(time.time())
snapshot_iso = datetime.datetime.fromtimestamp(
    snapshot_ts, tz=datetime.timezone.utc
).isoformat()

person_count = int(next(iter(bq.query(
    f'SELECT COUNT(*) AS n FROM {DS}.people WHERE composite_score IS NOT NULL'
).result())).n)

config = json.loads(Path('config.json').read_text())
config.setdefault('gcp', {})['scoring_snapshot_ts'] = snapshot_ts
config['gcp']['scoring_snapshot_iso'] = snapshot_iso
config['gcp']['scoring_person_count'] = person_count
Path('config.json').write_text(json.dumps(config, indent=2))

print()
print(f'✓ snapshot_ts={snapshot_ts} ({snapshot_iso})')
print(f'✓ scored {person_count} people')
print('Next: 15-deploy-agent-engine.ipynb — deploy ADK agents to Agent Engine')
